# 01 — Tensors: PyTorch's Core Data Structure

**Project:** AutoGrader  
**Purpose:** Foundation skill — understand and manipulate tensors from scratch  
**Covers:** Creation, reshaping, indexing, matrix ops, NumPy interop, GPU vs CPU

---

## Step 0 — Install PyTorch

Run this cell once. If PyTorch is already installed, it'll just confirm the version.

In [20]:
# Install PyTorch (CPU version — works on any machine including Mac)
# If you have an NVIDIA GPU, visit pytorch.org/get-started for the CUDA version
%pip install torch torchvision --quiet

import torch
import numpy as np

print(f"PyTorch version: {torch.__version__}")
print(f"NumPy version: {np.__version__}")
print(f"GPU available: {torch.cuda.is_available()}")
print(f"MPS (Apple Silicon) available: {torch.backends.mps.is_available()}")


[notice] A new release of pip is available: 24.3.1 -> 26.1.1
[notice] To update, run: python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
PyTorch version: 2.5.1
NumPy version: 1.26.4
GPU available: False
MPS (Apple Silicon) available: True


---
## Section 1 — Creating Tensors

There are 4 main ways to create a tensor. You need to know all of them.

In [21]:
# --- Way 1: From a Python list ---
# Think of this like wrapping your data in a PyTorch-aware container

scalar = torch.tensor(42)                        # 0D — single number
vector = torch.tensor([1.0, 2.0, 3.0])          # 1D — like a list
matrix = torch.tensor([[1, 2, 3],               # 2D — like a table
                        [4, 5, 6]])
cube   = torch.tensor([[[1, 2], [3, 4]],        # 3D — like a stack of tables
                        [[5, 6], [7, 8]]])

print("Scalar:", scalar, "| shape:", scalar.shape)
print("Vector:", vector, "| shape:", vector.shape)
print("Matrix:\n", matrix, "| shape:", matrix.shape)
print("Cube shape:", cube.shape)  # (2, 2, 2) = 2 tables, 2 rows, 2 cols

Scalar: tensor(42) | shape: torch.Size([])
Vector: tensor([1., 2., 3.]) | shape: torch.Size([3])
Matrix:
 tensor([[1, 2, 3],
        [4, 5, 6]]) | shape: torch.Size([2, 3])
Cube shape: torch.Size([2, 2, 2])


In [22]:
# --- Way 2: From NumPy array ---
# Common in real projects — you load data with NumPy/Pandas, then convert to tensor

np_array = np.array([1.5, 2.5, 3.5])
tensor_from_np = torch.from_numpy(np_array)

print("NumPy array:", np_array)
print("Tensor from NumPy:", tensor_from_np)
print("dtype:", tensor_from_np.dtype)  # float64 — inherits from NumPy

# IMPORTANT: from_numpy shares memory — changing one changes the other!
np_array[0] = 99
print("After changing NumPy array[0] to 99:")
print("NumPy:", np_array)
print("Tensor (also changed!):", tensor_from_np)  # shared memory

NumPy array: [1.5 2.5 3.5]
Tensor from NumPy: tensor([1.5000, 2.5000, 3.5000], dtype=torch.float64)
dtype: torch.float64
After changing NumPy array[0] to 99:
NumPy: [99.   2.5  3.5]
Tensor (also changed!): tensor([99.0000,  2.5000,  3.5000], dtype=torch.float64)


In [23]:
# --- Way 3: Built-in factory functions ---
# These are like shortcuts for common patterns

zeros = torch.zeros(3, 4)        # 3x4 matrix of zeros
ones  = torch.ones(2, 3)         # 2x3 matrix of ones
rand  = torch.rand(3, 3)         # random values between 0 and 1
randn = torch.randn(3, 3)        # random values from normal distribution (mean=0, std=1)
eye   = torch.eye(4)             # 4x4 identity matrix (diagonal = 1, rest = 0)
arange = torch.arange(0, 10, 2) # like range() → [0, 2, 4, 6, 8]

print("Zeros (3x4):\n", zeros)
print("\nOnes (2x3):\n", ones)
print("\nRandom (3x3):\n", rand)
print("\nArange (0 to 10, step 2):", arange)

Zeros (3x4):
 tensor([[0., 0., 0., 0.],
        [0., 0., 0., 0.],
        [0., 0., 0., 0.]])

Ones (2x3):
 tensor([[1., 1., 1.],
        [1., 1., 1.]])

Random (3x3):
 tensor([[0.6972, 0.8149, 0.1085],
        [0.6455, 0.7903, 0.1483],
        [0.1259, 0.1623, 0.5350]])

Arange (0 to 10, step 2): tensor([0, 2, 4, 6, 8])


In [24]:
# --- Way 4: Specifying data type (dtype) ---
# dtype matters a lot — wrong dtype = errors or wasted memory

float32 = torch.tensor([1.0, 2.0], dtype=torch.float32)  # default for neural nets
float64 = torch.tensor([1.0, 2.0], dtype=torch.float64)  # more precision, more memory
int32   = torch.tensor([1, 2, 3],  dtype=torch.int32)
bool_t  = torch.tensor([True, False, True], dtype=torch.bool)

print("float32:", float32, "| dtype:", float32.dtype)
print("float64:", float64, "| dtype:", float64.dtype)
print("int32:  ", int32,   "| dtype:", int32.dtype)
print("bool:   ", bool_t,  "| dtype:", bool_t.dtype)

# Rule of thumb: use float32 for neural nets (GPU optimized), float64 for scientific computing

float32: tensor([1., 2.]) | dtype: torch.float32
float64: tensor([1., 2.], dtype=torch.float64) | dtype: torch.float64
int32:   tensor([1, 2, 3], dtype=torch.int32) | dtype: torch.int32
bool:    tensor([ True, False,  True]) | dtype: torch.bool


---
## Section 2 — Shape, Reshape, and View

Shape is one of the most common sources of bugs in PyTorch. Get comfortable with it.

In [25]:
# --- Inspecting shape ---
t = torch.rand(2, 3, 4)  # 3D tensor: 2 blocks, 3 rows, 4 cols

print("Shape:", t.shape)          # torch.Size([2, 3, 4])
print("Dimensions:", t.ndim)      # 3
print("Total elements:", t.numel())  # 2*3*4 = 24
print("dtype:", t.dtype)
print("device:", t.device)        # cpu (or cuda:0 if on GPU)

Shape: torch.Size([2, 3, 4])
Dimensions: 3
Total elements: 24
dtype: torch.float32
device: cpu


In [26]:
# --- reshape vs view ---
# Both change shape. view() requires contiguous memory. reshape() handles both cases.
# Rule: always use reshape() unless you have a specific reason for view()

t = torch.arange(12)   # [0, 1, 2, ..., 11]
print("Original:", t, "| shape:", t.shape)

t_2d = t.reshape(3, 4)   # 3 rows, 4 cols
print("\nReshaped to (3,4):\n", t_2d)

t_2d_b = t.reshape(4, 3) # 4 rows, 3 cols
print("\nReshaped to (4,3):\n", t_2d_b)

# -1 means "figure it out automatically"
t_auto = t.reshape(2, -1)  # 2 rows, PyTorch calculates cols = 6
print("\nReshaped to (2,-1):", t_auto.shape)  # [2, 6]

Original: tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11]) | shape: torch.Size([12])

Reshaped to (3,4):
 tensor([[ 0,  1,  2,  3],
        [ 4,  5,  6,  7],
        [ 8,  9, 10, 11]])

Reshaped to (4,3):
 tensor([[ 0,  1,  2],
        [ 3,  4,  5],
        [ 6,  7,  8],
        [ 9, 10, 11]])

Reshaped to (2,-1): torch.Size([2, 6])


In [27]:
# --- squeeze and unsqueeze ---
# squeeze: remove dimensions of size 1
# unsqueeze: add a dimension of size 1
# You'll see this constantly when working with batch sizes in neural nets

t = torch.rand(1, 3, 1)   # has two "useless" dimensions of size 1
print("Original shape:", t.shape)          # [1, 3, 1]
print("After squeeze:", t.squeeze().shape) # [3]

t2 = torch.rand(3)         # 1D tensor
print("\nOriginal:", t2.shape)                  # [3]
print("unsqueeze(0):", t2.unsqueeze(0).shape)  # [1, 3] — add batch dimension at front
print("unsqueeze(1):", t2.unsqueeze(1).shape)  # [3, 1] — add dimension at end

Original shape: torch.Size([1, 3, 1])
After squeeze: torch.Size([3])

Original: torch.Size([3])
unsqueeze(0): torch.Size([1, 3])
unsqueeze(1): torch.Size([3, 1])


---
## Section 3 — Indexing and Slicing

Same syntax as NumPy. If you know NumPy slicing, you already know this.

In [28]:
t = torch.tensor([[10, 20, 30],
                   [40, 50, 60],
                   [70, 80, 90]])

print("Full tensor:\n", t)
print("\nRow 0:", t[0])            # first row → [10, 20, 30]
print("Row 1, Col 2:", t[1, 2])   # single element → 60
print("All rows, Col 1:", t[:, 1]) # entire second column → [20, 50, 80]
print("Rows 0-1:\n", t[0:2])       # first two rows
print("Last row:", t[-1])          # last row → [70, 80, 90]

Full tensor:
 tensor([[10, 20, 30],
        [40, 50, 60],
        [70, 80, 90]])

Row 0: tensor([10, 20, 30])
Row 1, Col 2: tensor(60)
All rows, Col 1: tensor([20, 50, 80])
Rows 0-1:
 tensor([[10, 20, 30],
        [40, 50, 60]])
Last row: tensor([70, 80, 90])


In [29]:
# --- Boolean / conditional indexing ---
# Very useful for filtering embeddings, masking, etc.

t = torch.tensor([1.0, -2.0, 3.0, -4.0, 5.0])

mask = t > 0
print("Mask (where t > 0):", mask)    # [True, False, True, False, True]
print("Positive values:", t[mask])     # [1., 3., 5.]

# In AutoGrader context: imagine filtering similarity scores above a threshold
similarity_scores = torch.tensor([0.82, 0.45, 0.91, 0.30, 0.76])
threshold = 0.70
high_similarity = similarity_scores[similarity_scores > threshold]
print("\nScores above 0.70:", high_similarity)

Mask (where t > 0): tensor([ True, False,  True, False,  True])
Positive values: tensor([1., 3., 5.])

Scores above 0.70: tensor([0.8200, 0.9100, 0.7600])


---
## Section 4 — Math and Matrix Operations

This is where tensors earn their keep. All neural net math is just these operations.

In [30]:
# --- Element-wise operations ---
# Operations applied to each element independently

a = torch.tensor([1.0, 2.0, 3.0])
b = torch.tensor([4.0, 5.0, 6.0])

print("a + b  :", a + b)          # [5, 7, 9]
print("a - b  :", a - b)          # [-3, -3, -3]
print("a * b  :", a * b)          # [4, 10, 18]  ← element-wise, NOT matrix multiply
print("a / b  :", a / b)          # [0.25, 0.4, 0.5]
print("a ** 2 :", a ** 2)         # [1, 4, 9]
print("sqrt(a):", torch.sqrt(a))  # [1, 1.41, 1.73]

a + b  : tensor([5., 7., 9.])
a - b  : tensor([-3., -3., -3.])
a * b  : tensor([ 4., 10., 18.])
a / b  : tensor([0.2500, 0.4000, 0.5000])
a ** 2 : tensor([1., 4., 9.])
sqrt(a): tensor([1.0000, 1.4142, 1.7321])


In [31]:
# --- Matrix multiplication — the most important operation in deep learning ---
# Every layer in a neural network is a matrix multiply

A = torch.tensor([[1.0, 2.0],
                   [3.0, 4.0]])   # 2x2

B = torch.tensor([[5.0, 6.0],
                   [7.0, 8.0]])   # 2x2

# Three equivalent ways to do matrix multiply:
print("torch.matmul(A, B):\n", torch.matmul(A, B))
print("A @ B:\n", A @ B)          # @ is the matrix multiply operator
print("torch.mm(A, B):\n", torch.mm(A, B))  # only works for 2D

# Use @ in practice — clean and readable

torch.matmul(A, B):
 tensor([[19., 22.],
        [43., 50.]])
A @ B:
 tensor([[19., 22.],
        [43., 50.]])
torch.mm(A, B):
 tensor([[19., 22.],
        [43., 50.]])


In [32]:
# --- Dot product and cosine similarity ---
# AutoGrader uses cosine similarity to compare answer embeddings

student_answer_embedding  = torch.tensor([0.6, 0.8, 0.1])
correct_answer_embedding  = torch.tensor([0.5, 0.9, 0.2])

# Dot product
dot = torch.dot(student_answer_embedding, correct_answer_embedding)
print("Dot product:", dot.item())

# Cosine similarity = dot product / (magnitude1 * magnitude2)
# Value between -1 and 1. Closer to 1 = more similar.
cos_sim = torch.nn.functional.cosine_similarity(
    student_answer_embedding.unsqueeze(0),
    correct_answer_embedding.unsqueeze(0)
)
print("Cosine similarity:", cos_sim.item())  # This is how RAG ranking works!

Dot product: 1.0399999618530273
Cosine similarity: 0.9866799116134644


In [33]:
# --- Aggregation operations ---

t = torch.tensor([[1.0, 2.0, 3.0],
                   [4.0, 5.0, 6.0]])

print("Sum (all):", t.sum().item())
print("Sum (per column):", t.sum(dim=0))   # sum down rows → [5, 7, 9]
print("Sum (per row):", t.sum(dim=1))      # sum across cols → [6, 15]
print("Mean:", t.mean().item())
print("Max:", t.max().item())
print("Argmax (index of max):", t.argmax().item())  # flattened index
print("Argmax per row:", t.argmax(dim=1))   # which col has max, per row

Sum (all): 21.0
Sum (per column): tensor([5., 7., 9.])
Sum (per row): tensor([ 6., 15.])
Mean: 3.5
Max: 6.0
Argmax (index of max): 5
Argmax per row: tensor([2, 2])


---
## Section 5 — Tensor vs NumPy Array

Key differences you must be able to explain in an interview.

In [34]:
# --- Converting between tensor and NumPy ---

# Tensor → NumPy
t = torch.tensor([1.0, 2.0, 3.0])
arr = t.numpy()   # shares memory if tensor is on CPU
print("Tensor to NumPy:", arr, type(arr))

# NumPy → Tensor
arr2 = np.array([4.0, 5.0, 6.0])
t2 = torch.from_numpy(arr2)     # shares memory
t3 = torch.tensor(arr2)         # copies memory (safer)
print("NumPy to Tensor (shared):", t2)
print("NumPy to Tensor (copy):", t3)

Tensor to NumPy: [1. 2. 3.] <class 'numpy.ndarray'>
NumPy to Tensor (shared): tensor([4., 5., 6.], dtype=torch.float64)
NumPy to Tensor (copy): tensor([4., 5., 6.], dtype=torch.float64)


In [35]:
# --- The 4 key differences: Tensor vs NumPy ---

print("=" * 50)
print("DIFFERENCE 1: GPU Support")
print("=" * 50)
t = torch.tensor([1.0, 2.0, 3.0])
print("Tensor device:", t.device)          # cpu
# NumPy arrays always stay on CPU. Period.
# Tensors can move to GPU: t.to('cuda') or t.cuda()
print("NumPy: always CPU only")

print("\n" + "=" * 50)
print("DIFFERENCE 2: Gradient Tracking (Autograd)")
print("=" * 50)
t_grad = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)
print("requires_grad:", t_grad.requires_grad)  # True
print("NumPy: no gradient tracking — can't train neural nets")

print("\n" + "=" * 50)
print("DIFFERENCE 3: Default dtype")
print("=" * 50)
np_arr = np.array([1.0, 2.0])    # float64 by default
torch_t = torch.tensor([1.0, 2.0])  # float32 by default
print("NumPy default dtype:", np_arr.dtype)   # float64
print("PyTorch default dtype:", torch_t.dtype) # float32 (GPU optimized)

print("\n" + "=" * 50)
print("DIFFERENCE 4: Deep Learning Ecosystem")
print("=" * 50)
print("Tensors plug directly into nn.Module, DataLoader, optimizers")
print("NumPy arrays need conversion first")

DIFFERENCE 1: GPU Support
Tensor device: cpu
NumPy: always CPU only

DIFFERENCE 2: Gradient Tracking (Autograd)
requires_grad: True
NumPy: no gradient tracking — can't train neural nets

DIFFERENCE 3: Default dtype
NumPy default dtype: float64
PyTorch default dtype: torch.float32

DIFFERENCE 4: Deep Learning Ecosystem
Tensors plug directly into nn.Module, DataLoader, optimizers
NumPy arrays need conversion first


---
## Section 6 — GPU vs CPU

The device a tensor lives on determines where the computation happens.

In [36]:
# --- Detecting available device ---
# This is the standard pattern used in every real PyTorch project

if torch.cuda.is_available():
    device = torch.device("cuda")       # NVIDIA GPU
    print("Using NVIDIA GPU:", torch.cuda.get_device_name(0))
elif torch.backends.mps.is_available():
    device = torch.device("mps")        # Apple Silicon (M1/M2/M3)
    print("Using Apple Silicon MPS")
else:
    device = torch.device("cpu")        # fallback
    print("Using CPU")

print("Device selected:", device)

Using Apple Silicon MPS
Device selected: mps


In [37]:
# --- Moving tensors between devices ---

# Create on CPU (default)
t_cpu = torch.rand(3, 3)
print("On CPU:", t_cpu.device)

# Move to whichever device we have
t_device = t_cpu.to(device)
print("Moved to device:", t_device.device)

# Move back to CPU (needed before converting to NumPy)
t_back = t_device.cpu()
print("Back on CPU:", t_back.device)

# IMPORTANT RULE: Both tensors in an operation must be on the SAME device
# This will error if devices differ:
# result = t_cpu + t_device  ← RuntimeError!

On CPU: cpu
Moved to device: mps:0
Back on CPU: cpu


In [38]:
# --- Why GPU matters for AutoGrader ---
# When computing embeddings for 1000 student answers simultaneously:

import time

# Simulate batch of embeddings (1000 answers, 768-dim embedding each)
batch_size = 1000
embed_dim  = 768

embeddings = torch.rand(batch_size, embed_dim)

# CPU computation
start = time.time()
result_cpu = embeddings @ embeddings.T  # similarity matrix
cpu_time = time.time() - start
print(f"CPU time for {batch_size}x{batch_size} similarity matrix: {cpu_time:.4f}s")

# GPU computation (if available)
if device.type != "cpu":
    embeddings_gpu = embeddings.to(device)
    torch.cuda.synchronize() if device.type == "cuda" else None
    start = time.time()
    result_gpu = embeddings_gpu @ embeddings_gpu.T
    torch.cuda.synchronize() if device.type == "cuda" else None
    gpu_time = time.time() - start
    print(f"GPU time: {gpu_time:.4f}s")
    print(f"Speedup: {cpu_time/gpu_time:.1f}x faster")
else:
    print("No GPU available — in production, this step is 10-100x faster on GPU")

CPU time for 1000x1000 similarity matrix: 0.0094s
GPU time: 0.0356s
Speedup: 0.3x faster


---
## Section 7 — Quick Reference Cheatsheet

Everything you need in one place.

In [39]:
# ================================================================
# TENSOR CHEATSHEET — AutoGrader / PyTorch Foundations
# ================================================================

# CREATION
# torch.tensor([1,2,3])          ← from list
# torch.from_numpy(arr)          ← from NumPy (shared memory)
# torch.tensor(arr)              ← from NumPy (copy)
# torch.zeros(m, n)              ← zeros
# torch.ones(m, n)               ← ones
# torch.rand(m, n)               ← uniform random [0,1]
# torch.randn(m, n)              ← normal random (mean=0, std=1)
# torch.eye(n)                   ← identity matrix
# torch.arange(start, end, step) ← like range()

# INSPECTION
# t.shape                        ← dimensions
# t.ndim                         ← number of dimensions
# t.numel()                      ← total number of elements
# t.dtype                        ← data type
# t.device                       ← cpu or cuda

# RESHAPING
# t.reshape(m, n)                ← reshape (use -1 to auto-calculate)
# t.squeeze()                    ← remove size-1 dimensions
# t.unsqueeze(dim)               ← add a size-1 dimension
# t.T or t.transpose(0,1)        ← transpose

# INDEXING
# t[0]                           ← first row
# t[1, 2]                        ← row 1, col 2
# t[:, 1]                        ← all rows, col 1
# t[t > 0]                       ← boolean mask

# MATH
# a + b, a - b, a * b, a / b     ← element-wise
# a @ b or torch.matmul(a, b)    ← matrix multiply
# torch.dot(a, b)                ← dot product (1D only)
# t.sum(), t.mean(), t.max()     ← aggregation
# t.sum(dim=0)                   ← aggregation along dimension

# DEVICE
# t.to(device)                   ← move to device
# t.cpu()                        ← move to CPU
# t.device                       ← check current device

# NUMPY INTEROP
# t.numpy()                      ← tensor → NumPy (CPU only)
# torch.from_numpy(arr)          ← NumPy → tensor (shared)
# torch.tensor(arr)              ← NumPy → tensor (copy)

print("Cheatsheet loaded. You're ready.")

Cheatsheet loaded. You're ready.


---
## ✅ Proof Checklist

Run through this to confirm you've nailed the skill:

- [ ] Created tensors from a list, NumPy array, and factory functions
- [ ] Inspected shape, dtype, device, numel
- [ ] Reshaped using reshape() with -1 auto-calculation
- [ ] Used squeeze() and unsqueeze()
- [ ] Indexed rows, columns, and used boolean mask
- [ ] Did element-wise math and matrix multiply with @
- [ ] Computed cosine similarity (relevant to AutoGrader RAG)
- [ ] Explained 4 differences between tensor and NumPy array
- [ ] Detected device and moved tensor between CPU/GPU

**Next skill:** Autograd — automatic differentiation (`02_autograd.ipynb`)